<a href="https://colab.research.google.com/github/Nakib-Nasrullah/Dengue_Research/blob/main/Dengue_best_math_equa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Data handling and visualization
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [2]:
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

df = pd.read_csv('/content/drive/MyDrive/Datasets/Dengue_final_dataset_updated.csv')
print("Dataset loaded successfully!!")
print(f"\n Dataset shape: {df.shape}")

Mounted at /content/drive
Dataset loaded successfully!!

 Dataset shape: (23360, 23)


In [4]:
# Convert date column
df['date'] = pd.to_datetime(df['Date'])

# Sort by date (CRITICAL)
df = df.sort_values('date').reset_index(drop=True)

# Basic check
print(df.head())

       Date    District  Max_Temp  Min_Temp  Avg_Temp  Humidity  Rainfall  \
0  1/1/2025       Dhaka     30.04     20.69     25.36     62.83      2.46   
1  1/1/2025   Netrokona     25.61     19.34     22.47     67.60      2.78   
2  1/1/2025   Sirajganj     25.02     17.17     21.09     54.53      0.54   
3  1/1/2025   Madaripur     29.36     20.40     24.88     50.32      1.64   
4  1/1/2025  Jhalokathi     35.23     29.54     32.38     55.68      1.29   

   Rainfall_Lag_7  Rainfall_Lag_14  Rainfall_Lag_21  ...  Temp_Lag_30  \
0             0.0              0.0              0.0  ...          0.0   
1             0.0              0.0              0.0  ...          0.0   
2             0.0              0.0              0.0  ...          0.0   
3             0.0              0.0              0.0  ...          0.0   
4             0.0              0.0              0.0  ...          0.0   

   Humidity_Lag_7  Humidity_Lag_14  Humidity_Lag_21  Humidity_Lag_30  \
0             0.0         

# **Create Gamma weights**

In [5]:
from scipy.stats import gamma

def create_weights(lag_min=7, lag_max=35, mean=18, sd=6):
    tau = np.arange(lag_min, lag_max + 1)

    # Convert mean & sd → gamma parameters
    shape = (mean / sd) ** 2
    scale = (sd ** 2) / mean

    weights = gamma.pdf(tau, a=shape, scale=scale)

    # Normalize
    weights = weights / weights.sum()

    return tau, weights


# **Compute infectious pressure**

In [6]:
def compute_infectious_pressure(cases, tau, weights):
    I = np.zeros(len(cases))

    for t in range(len(cases)):
        val = 0
        for i, lag in enumerate(tau):
            if t - lag >= 0:
                val += weights[i] * cases[t - lag]
        I[t] = val

    return I

# **Apply to dataset**

In [8]:
tau, weights = create_weights(mean=18, sd=6)

df['I_t'] = compute_infectious_pressure(df['Dengue_case'].values, tau, weights)

# Avoid log(0)
df['log_I'] = np.log(df['I_t'] + 1)

# **Create rainfall & temperature lag features**
We now build distributed lag features.

In [9]:
def create_lags(df, col, max_lag):
    for lag in range(0, max_lag + 1):
        df[f"{col}_lag{lag}"] = df[col].shift(lag)
    return df

In [12]:
df = create_lags(df, 'Rainfall', 60)
df = create_lags(df, 'Avg_Temp', 30)

# **Drop NA rows (due to lagging)**

In [13]:
df_model = df.dropna().copy()

# **Build rainfall & temperature summaries**

In [15]:
def lag_window_sum(df, col, start, end):
    cols = [f"{col}_lag{i}" for i in range(start, end + 1)]
    return df[cols].sum(axis=1)

# Rainfall windows
df_model['rain_0_14'] = lag_window_sum(df_model, 'Rainfall', 0, 14)
df_model['rain_15_30'] = lag_window_sum(df_model, 'Rainfall', 15, 30)
df_model['rain_31_60'] = lag_window_sum(df_model, 'Rainfall', 31, 60)

# Temperature windows (mean instead of sum)
def lag_window_mean(df, col, start, end):
    cols = [f"{col}_lag{i}" for i in range(start, end + 1)]
    return df[cols].mean(axis=1)

df_model['temp_0_14'] = lag_window_mean(df_model, 'Avg_Temp', 0, 14)
df_model['temp_15_30'] = lag_window_mean(df_model, 'Avg_Temp', 15, 30)

# **Interaction terms (VERY IMPORTANT)**

In [16]:
df_model['rain_I_interaction'] = df_model['rain_15_30'] * df_model['log_I']
df_model['temp_I_interaction'] = df_model['temp_0_14'] * df_model['log_I']


# **Build the full model**
We use Negative Binomial regression.

In [18]:
import statsmodels.api as sm

# Define features
X = df_model[[
    'rain_0_14',
    'rain_15_30',
    'rain_31_60',
    'temp_0_14',
    'temp_15_30',
    'log_I',
    'rain_I_interaction',
    'temp_I_interaction'
]]

# Add intercept
X = sm.add_constant(X)

# Target
y = df_model['Dengue_case']

# Fit Negative Binomial model
model = sm.GLM(y, X, family=sm.families.NegativeBinomial())
results = model.fit()

print(results.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:            Dengue_case   No. Observations:                23300
Model:                            GLM   Df Residuals:                    23291
Model Family:        NegativeBinomial   Df Model:                            8
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -50203.
Date:                Thu, 02 Apr 2026   Deviance:                       55003.
Time:                        05:29:36   Pearson chi2:                 2.11e+05
No. Iterations:                    14   Pseudo R-squ. (CS):             0.5782
Covariance Type:            nonrobust                                         
                         coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------
const                 -0.3122      0

/usr/local/lib/python3.12/dist-packages/statsmodels/genmod/families/family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "


# ***Fit optimal infectious weights*** (core research step)
Now we optimize mean & SD of weights.
# **Grid search for best weights**

In [21]:
best_aic = np.inf
best_params = None

for mean in range(14, 25):        # Bangladesh range
    for sd in range(4, 9):

        tau, weights = create_weights(mean=mean, sd=sd)
        df['I_t'] = compute_infectious_pressure(df['Dengue_case'].values, tau, weights)
        df['log_I'] = np.log(df['I_t'] + 1)

        df_temp = df.dropna().copy()

        # Re-calculate summary features for df_temp
        df_temp['rain_0_14'] = lag_window_sum(df_temp, 'Rainfall', 0, 14)
        df_temp['rain_15_30'] = lag_window_sum(df_temp, 'Rainfall', 15, 30)
        df_temp['rain_31_60'] = lag_window_sum(df_temp, 'Rainfall', 31, 60)
        df_temp['temp_0_14'] = lag_window_mean(df_temp, 'Avg_Temp', 0, 14)
        df_temp['temp_15_30'] = lag_window_mean(df_temp, 'Avg_Temp', 15, 30)

        X_temp = df_temp[[
            'rain_0_14',
            'rain_15_30',
            'rain_31_60',
            'temp_0_14',
            'temp_15_30',
            'log_I'
        ]]
        X_temp = sm.add_constant(X_temp)
        y_temp = df_temp['Dengue_case']

        try:
            model_temp = sm.GLM(y_temp, X_temp, family=sm.families.NegativeBinomial())
            result_temp = model_temp.fit()

            if result_temp.aic < best_aic:
                best_aic = result_temp.aic
                best_params = (mean, sd)

        except:
            continue

print("Best mean, sd:", best_params)
print("Best AIC:", best_aic)

/usr/local/lib/python3.12/dist-packages/statsmodels/genmod/families/family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "
/usr/local/lib/python3.12/dist-packages/statsmodels/genmod/families/family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "
/usr/local/lib/python3.12/dist-packages/statsmodels/genmod/families/family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "
/usr/local/lib/python3.12/dist-packages/statsmodels/genmod/families/family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha no

Best mean, sd: (21, 8)
Best AIC: 101543.55864277169
